# Pré-requis : Dataset, Tête de classification, Test split commun

In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report
from sklearn.model_selection import train_test_split
from constants import FEATURES_DIR
import os
import mlflow
import mlflow.pytorch

# Config
# ====
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 64
EPOCHS = 30
LR = 1e-3
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

# ====
# Dataset
# ====
class FeatureDataset(Dataset): # ajouter si la data est pseudo ou strong labellisée
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# ====
# ResNet50 head (classification) on top of ResNet50 embeddings (avgpool)
# ====
class ResNet50Head(nn.Module): ## nn.Module => héritage réseau de neuronnes pytorch
    """
    Classification head qui prend en entrée des features extraites par ResNet50
    """
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 2)
        )
    
    def forward(self, x):
        return self.net(x)

# ====
# CRÉATION DU TEST STRONG COMMUN (pour comparaison équitable)
# ====
print("\n" + "=" * 50)
print("CRÉATION DU TEST STRONG COMMUN")
print("=" * 50)

X_cancer_strong = np.load(FEATURES_DIR / "features_cancer_avgpool.npy")
X_normal_strong = np.load(FEATURES_DIR / "features_normal_avgpool.npy")

X_strong = np.vstack([X_cancer_strong, X_normal_strong])
y_strong = np.array([1] * len(X_cancer_strong) + [0] * len(X_normal_strong))

# Test Split UNIQUE sur les STRONG labels : 80% train / 20% test
X_s_train, X_s_test, y_s_train, y_s_test = train_test_split(
    X_strong,
    y_strong,
    test_size=0.2,
    random_state=SEED,
    stratify=y_strong
)

print(f"Strong train: {len(y_s_train)} | Strong test: {len(y_s_test)}")
print(f"Distribution test: {np.bincount(y_s_test)}")


CRÉATION DU TEST STRONG COMMUN
Strong train: 80 | Strong test: 20
Distribution test: [10 10]


# Approche Semi-supervisée

In [2]:
# ====
# Approche Semi-supervisée
# ====

# MLflow Config
MLFLOW_EXPERIMENT_NAME = "resnet50-weaklabels"

# ====
# # Load data
# ====
print("\nChargement des features.")
files = [f for f in os.listdir(FEATURES_DIR) if f.endswith(".npy")]
X = np.load(FEATURES_DIR / "features_unlabeled_avgpool.npy")
y = np.load(FEATURES_DIR / "weak_labels.npy")

print(f"\nSamples weak-labeled: {len(y)}")
print(f"X shape: {X.shape}")
print(f"Distribution: {np.bincount(y)}")

# ====
# # Train / Val split
# ====
dataset = FeatureDataset(X, y)
train_size = int(0.8 * len(dataset)) # 80% des données pour l'entrainement
val_size = len(dataset) - train_size # 20% des données pour la validation
train_ds, val_ds = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True) # shuffle: on s'assure qu'il n'y ait pas un ordre "caché" que le modèle pourrait apprendre
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)

# ====
# # Training setup
# ====
model = ResNet50Head(input_dim=X.shape[1]).to(DEVICE) # shape[1] sur tableau numpy = nombre de colonnes
criterion = nn.CrossEntropyLoss() # mesure à quel point le modèle se trompe
optimizer = torch.optim.Adam(model.parameters(), lr=LR) # ajuste les poids du modèle pour réduire l'erreur

# ====
# # MLflow Run
# ====
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

with mlflow.start_run():
    # Log des hyperparamètres
    mlflow.log_params({
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "lr": LR,
        "seed": SEED,
        "device": DEVICE
    })
    
    print(f"\nTraining on {DEVICE}.\n")
    
    for epoch in range(EPOCHS):
        model.train()
        train_loss_sum = 0.0
        
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            
            optimizer.zero_grad()
            preds = model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()
            
            train_loss_sum += loss.item()
        
        # Validation
        model.eval()
        y_true, y_pred = [], []
        
        with torch.no_grad(): # no_grad() = inférence seule, pas d'entrainement
            for xb, yb in val_loader:
                xb = xb.to(DEVICE)
                preds = model(xb).argmax(1).cpu().numpy()
                y_pred.extend(preds)
                y_true.extend(yb.numpy())
        
        # Calcul des métriques
        acc = accuracy_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred)
        prec = precision_score(y_true, y_pred, zero_division=0)
        rec = recall_score(y_true, y_pred, zero_division=0)
        avg_train_loss = train_loss_sum / len(train_loader)
        
        # Sauvegarde MLflow par epoch
        mlflow.log_metric("train_loss", avg_train_loss, step=epoch)
        mlflow.log_metric("val_acc", acc, step=epoch)
        mlflow.log_metric("val_f1", f1, step=epoch)
        mlflow.log_metric("val_precision", prec, step=epoch)
        mlflow.log_metric("val_recall", rec, step=epoch)
        
        print(f"Epoch {epoch+1:02d} | Loss={avg_train_loss:.4f} | Acc={acc:.4f}")
    
    # ====
    # Final report (weak val)
    # ====
    print("\n" + "=" * 50)
    print("Classification report (weak labels) [val split]:")
    print("=" * 50)
    report_weak = classification_report(y_true, y_pred, target_names=["Normal", "Cancer"])
    print(report_weak)
    mlflow.log_text(report_weak, "reports/weak_labels_report.txt")
    
    # ====
    # TEST DE VÉRITÉ: Évaluation sur le TEST STRONG COMMUN
    # ====
    print("\n" + "=" * 50)
    print("TEST DE VÉRITÉ: Évaluation sur le TEST STRONG COMMUN")
    print("=" * 50)
    
    model.eval()
    X_s_test_tensor = torch.tensor(X_s_test, dtype=torch.float32).to(DEVICE)
    
    with torch.no_grad():
        outputs = model(X_s_test_tensor)
        y_pred_strong = outputs.argmax(1).cpu().numpy()
    
    # Métriques Strong (TEST COMMUN)
    acc_s = accuracy_score(y_s_test, y_pred_strong)
    f1_s = f1_score(y_s_test, y_pred_strong)
    prec_s = precision_score(y_s_test, y_pred_strong, zero_division=0)
    rec_s = recall_score(y_s_test, y_pred_strong, zero_division=0)
    
    mlflow.log_metrics({
        "strong_test_acc": acc_s,
        "strong_test_f1": f1_s,
        "strong_test_precision": prec_s,
        "strong_test_recall": rec_s
    })
    
    report_strong = classification_report(
        y_s_test,
        y_pred_strong,
        target_names=["Normal", "Cancer"]
    )
    
    print(report_strong)
    mlflow.log_text(report_strong, "reports/strong_test_report.txt")
    
    # Log du modèle dans MLflow
    mlflow.pytorch.log_model(model, "model")


Chargement des features.

Samples weak-labeled: 1406
X shape: (1406, 2048)
Distribution: [ 354 1052]


2026/01/13 13:18:32 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet




Training on cpu.

Epoch 01 | Loss=0.4582 | Acc=0.9752
Epoch 02 | Loss=0.1484 | Acc=0.9823
Epoch 03 | Loss=0.0673 | Acc=0.9681
Epoch 04 | Loss=0.0620 | Acc=0.9894
Epoch 05 | Loss=0.0388 | Acc=0.9858
Epoch 06 | Loss=0.0357 | Acc=0.9787
Epoch 07 | Loss=0.0388 | Acc=0.9858
Epoch 08 | Loss=0.0450 | Acc=0.9752
Epoch 09 | Loss=0.0342 | Acc=0.9894
Epoch 10 | Loss=0.0275 | Acc=0.9858
Epoch 11 | Loss=0.0335 | Acc=0.9894
Epoch 12 | Loss=0.0316 | Acc=0.9929
Epoch 13 | Loss=0.0185 | Acc=0.9858
Epoch 14 | Loss=0.0172 | Acc=0.9752
Epoch 15 | Loss=0.0190 | Acc=0.9752
Epoch 16 | Loss=0.0096 | Acc=0.9894
Epoch 17 | Loss=0.0183 | Acc=0.9894
Epoch 18 | Loss=0.0390 | Acc=0.9894
Epoch 19 | Loss=0.0250 | Acc=0.9894
Epoch 20 | Loss=0.0183 | Acc=0.9716
Epoch 21 | Loss=0.0187 | Acc=0.9894
Epoch 22 | Loss=0.0185 | Acc=0.9894
Epoch 23 | Loss=0.0208 | Acc=0.9823
Epoch 24 | Loss=0.0292 | Acc=0.9894
Epoch 25 | Loss=0.0275 | Acc=0.9823
Epoch 26 | Loss=0.0168 | Acc=0.9858
Epoch 27 | Loss=0.0169 | Acc=0.9858
Epoch 28 

2026/01/13 13:18:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Epoch 30 | Loss=0.0167 | Acc=0.9858

Classification report (weak labels) [val split]:
              precision    recall  f1-score   support

      Normal       0.96      0.99      0.97        67
      Cancer       1.00      0.99      0.99       215

    accuracy                           0.99       282
   macro avg       0.98      0.99      0.98       282
weighted avg       0.99      0.99      0.99       282


TEST DE VÉRITÉ: Évaluation sur le TEST STRONG COMMUN
              precision    recall  f1-score   support

      Normal       1.00      0.90      0.95        10
      Cancer       0.91      1.00      0.95        10

    accuracy                           0.95        20
   macro avg       0.95      0.95      0.95        20
weighted avg       0.95      0.95      0.95        20

🏃 View run calm-zebra-196 at: http://mlflow:5000/#/experiments/136364928939801491/runs/2e4416d4cd3b4bb8a43219519600eb3b
🧪 View experiment at: http://mlflow:5000/#/experiments/136364928939801491


# Approche supervisée

In [3]:
# ====
# MLflow Config (Supervised)
# ====
MLFLOW_EXPERIMENT_NAME_SUP = "resnet50-supervised"
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME_SUP)

with mlflow.start_run():
    # 1. Utilisation du MÊME split strong que l'approche semi-supervisée
    # (défini une seule fois en amont pour une comparaison équitable)
    
    # Log des hyperparamètres + infos split
    mlflow.log_params({
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "lr": LR,
        "seed": SEED,
        "device": DEVICE,
        "split": "train_test_split",
        "train_ratio": 0.8,
        "test_ratio": 0.2,
        "stratify": True,
        "input_dim": int(X.shape[1]),
        "n_train_strong": int(len(y_s_train)),
        "n_test_strong": int(len(y_s_test)),
    })
    
    # Créer Les Loaders pour La baseline
    train_loader_s = DataLoader(
        FeatureDataset(X_s_train, y_s_train),
        batch_size=BATCH_SIZE,
        shuffle=True
    ) # shuffle: on s'assure qu'il n'y ait pas un ordre "caché" que Le modèle pourrait apprendre
    test_tensor_s = torch.tensor(X_s_test, dtype=torch.float32).to(DEVICE)
    
    # Resnet50
    model_baseline = ResNet50Head(input_dim=X.shape[1]).to(DEVICE) # shape[1] sur tableau numpy = nombre de colonnes
    optimizer_b = torch.optim.Adam(model_baseline.parameters(), lr=LR) # ajuste les poids du modèle pour réduire l'erreur
    
    # Entrainement uniquement sur Le TRAIN Strong
    for epoch in range(EPOCHS):
        model_baseline.train()
        train_loss_sum = 0.0
        train_n_batches = 0
        
        for xb, yb in train_loader_s:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            
            optimizer_b.zero_grad()
            preds = model_baseline(xb)
            loss = criterion(preds, yb) # mesure à quel point Le modèle se trompe
            loss.backward()
            optimizer_b.step()
            
            train_loss_sum += loss.item()
            train_n_batches += 1
        
        avg_train_loss = train_loss_sum / max(train_n_batches, 1)
        
        # (optionnel mais utile): évaluer sur Le TEST à chaque epoch pour suivre la progression
        model_baseline.eval()
        with torch.no_grad(): # no_grad() = inférence seule, pas d'entrainement
            y_pred_epoch = model_baseline(test_tensor_s).argmax(1).cpu().numpy()
        
        acc = accuracy_score(y_s_test, y_pred_epoch)
        f1 = f1_score(y_s_test, y_pred_epoch)
        prec = precision_score(y_s_test, y_pred_epoch, zero_division=0)
        rec = recall_score(y_s_test, y_pred_epoch, zero_division=0)
        
        # Log MLflow par epoch
        mlflow.log_metric("train_loss", avg_train_loss, step=epoch)
        mlflow.log_metric("test_acc", acc, step=epoch)
        mlflow.log_metric("test_f1", f1, step=epoch)
        mlflow.log_metric("test_precision", prec, step=epoch)
        mlflow.log_metric("test_recall", rec, step=epoch)
        
        print(f"Epoch {epoch+1:02d} | Loss={avg_train_loss:.4f} | Acc={acc:.4f}")
    
    # 5. Évaluation sur le TEST Strong (données jamais vues)
    model_baseline.eval()
    with torch.no_grad():
        y_pred_baseline = model_baseline(test_tensor_s).argmax(1).cpu().numpy()
    
    print("\nRapport Baseline sur données INCONNUES:")
    report_sup = classification_report(
        y_s_test, y_pred_baseline,
        target_names=['Normal', 'Cancer'],
        digits=4
    )
    print(report_sup)
    
    # Log artifact: report texte
    mlflow.log_text(report_sup, "reports/supervised_strong_test_report.txt")
    
    # Log métriques finales (en plus de celles par epoch)
    mlflow.log_metrics({
        "final_test_acc": accuracy_score(y_s_test, y_pred_baseline),
        "final_test_f1": f1_score(y_s_test, y_pred_baseline),
        "final_test_precision": precision_score(y_s_test, y_pred_baseline, zero_division=0),
        "final_test_recall": recall_score(y_s_test, y_pred_baseline, zero_division=0)
    })
    
    # (optionnel) sauvegarde + Log du modèle
    save_path_sup = FEATURES_DIR / "resnet50_head_supervised_trained.pth"
    torch.save(model_baseline.state_dict(), save_path_sup)
    print(f"\nModèle supervisé sauvegardé: {save_path_sup}")
    
    mlflow.log_artifact(str(save_path_sup), "weights")
    mlflow.pytorch.log_model(model_baseline, "model")

Epoch 01 | Loss=0.6588 | Acc=0.9000
Epoch 02 | Loss=0.5964 | Acc=0.9000
Epoch 03 | Loss=0.4411 | Acc=0.9000
Epoch 04 | Loss=0.4092 | Acc=0.9000
Epoch 05 | Loss=0.2242 | Acc=0.9500
Epoch 06 | Loss=0.2542 | Acc=0.9500
Epoch 07 | Loss=0.1633 | Acc=0.9500
Epoch 08 | Loss=0.2100 | Acc=0.9500
Epoch 09 | Loss=0.1412 | Acc=0.8500
Epoch 10 | Loss=0.1497 | Acc=0.8500
Epoch 11 | Loss=0.1140 | Acc=0.9000
Epoch 12 | Loss=0.1566 | Acc=0.9500
Epoch 13 | Loss=0.0744 | Acc=0.9500
Epoch 14 | Loss=0.1444 | Acc=0.9500
Epoch 15 | Loss=0.0610 | Acc=0.9000
Epoch 16 | Loss=0.0732 | Acc=0.8500
Epoch 17 | Loss=0.1152 | Acc=0.9000
Epoch 18 | Loss=0.1046 | Acc=0.9500
Epoch 19 | Loss=0.0701 | Acc=0.9000
Epoch 20 | Loss=0.0633 | Acc=0.9500
Epoch 21 | Loss=0.0273 | Acc=0.9500
Epoch 22 | Loss=0.0355 | Acc=0.9500
Epoch 23 | Loss=0.1269 | Acc=0.9500
Epoch 24 | Loss=0.1000 | Acc=0.9500
Epoch 25 | Loss=0.0242 | Acc=0.9500
Epoch 26 | Loss=0.0093 | Acc=0.9500
Epoch 27 | Loss=0.0061 | Acc=0.9500
Epoch 28 | Loss=0.0409 | Acc

2026/01/13 13:18:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Epoch 30 | Loss=0.0373 | Acc=0.9500

Rapport Baseline sur données INCONNUES:
              precision    recall  f1-score   support

      Normal     1.0000    0.9000    0.9474        10
      Cancer     0.9091    1.0000    0.9524        10

    accuracy                         0.9500        20
   macro avg     0.9545    0.9500    0.9499        20
weighted avg     0.9545    0.9500    0.9499        20


Modèle supervisé sauvegardé: ../.data/features/resnet50_head_supervised_trained.pth
🏃 View run wistful-kite-127 at: http://mlflow:5000/#/experiments/881879557287879727/runs/3fee5fbee29f407ba293dec6cef5fc46
🧪 View experiment at: http://mlflow:5000/#/experiments/881879557287879727
